# A 4096-Token Window and Three Document Lengths

**Topic:** Sliding-window attention &nbsp;·&nbsp; **Difficulty:** Medium

Sliding-window attention lets each token see `w` neighbours. Stack `L` layers and
information can travel `L * (w - 1)` positions, the same receptive-field argument
as a CNN. That is the defence of window attention, and it is worth checking
whether the arithmetic supports the claim at the sizes people actually use.

Compute the reachable distance, and the number of layers a document really
needs.

## The data (created in the next cell)

- Window `w = 4096`, `L = 32` layers (Mistral-like).
- Document lengths `[8192, 32768, 131072]`.
- Also compute the exact reachability by propagating a boolean mask, to confirm
  the formula.

## Your task

With **NumPy only**:

1. `theoretical_reach`: `L * (w - 1)` for causal sliding-window attention,
   where each layer extends the reach by `w - 1`.
2. Verify empirically at a small scale: with `n = 512`, `w_small = 16`, propagate
   a reachability mask layer by layer for 32 layers and record, for the last
   token, the earliest position it can reach at each layer.
3. `layers_needed`: for each document length, the smallest `L` such that
   `L * (w - 1) >= length`.
4. Set `answers` to a dict with `"theoretical_reach"`, `"empirical_reach"`
   (length-32, from the small-scale check), `"layers_needed"` (3 values), and
   `"formula_matches"`: `1.0` if, at every layer `k` from 1 to 32, the
   empirical reach equals `max(0, (n_small - 1) - k * (w_small - 1))`, that
   is, the formula's distance measured back from the last token and clipped
   at position 0.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the empirical propagation matches the formula.

In [ ]:
import numpy as np

w, L = 4096, 32
doc_lengths = [8192, 32768, 131072]
n_small, w_small = 512, 16
print(f"window {w}, {L} layers -> theoretical reach {L * (w - 1):,} tokens")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("theoretical_reach", [], 131040.0, -21941.07746129197, 1e-06),
    ("empirical_reach", [32], 8432.0, 383.5180568469552, 1e-06),
    ("layers_needed", [3], 45.0, -48.82912806907011, 1e-06),
    ("formula_matches", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "formula_matches" in answers:
    _m = float(answers["formula_matches"]) == 1.0
    print(f"  {'empirical = formula':<24} {'ok' if _m else 'wrong'}")
    _ok = _ok and _m

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Each layer lets a token pull from the `w - 1` positions behind it, and stacking
composes those reaches additively, exactly like receptive fields in a CNN.
Mistral's 4096-window, 32-layer configuration reaches 131,040 tokens, which
comfortably covers its 32k context. The catch the arithmetic does
not show: information reaching a token after 32 hops has been mixed with
everything else 32 times, so *reachable* is not the same as *retrievable*, which
is why long-context benchmarks still separate the two.

```python
import numpy as np

theoretical_reach = float(L * (w - 1))

reach = np.zeros((n_small,), dtype=bool)
reach[-1] = True
half = w_small - 1
empirical = []
cur = reach.copy()
for _ in range(32):
    new = cur.copy()
    idx = np.flatnonzero(cur)
    for i in idx:
        lo = max(0, i - half)
        new[lo:i + 1] = True
    cur = new
    empirical.append(float(np.flatnonzero(cur).min()))
empirical = np.array(empirical)

layers_needed = np.array([float(int(np.ceil(dl / (w - 1)))) for dl in doc_lengths])

expected = np.array([max(0.0, (n_small - 1) - (i + 1) * half) for i in range(32)])
matches = float(np.allclose(empirical, expected))

answers = {
    "theoretical_reach": theoretical_reach,
    "empirical_reach": empirical,
    "layers_needed": layers_needed,
    "formula_matches": matches,
}
```